# Praktikum Kecerdasan Artifisial Lanjut


---

## Bab 5. Klasifikasi Decision Tree dan XGBoost


### Decision Tree

#### 1) Import Data

Praktikum kali ini menggunakan dataset [Car Evaluation Dataset](https://archive.ics.uci.edu/ml/datasets/Car+Evaluation) dari UCI Machine Learning Repository. Dataset ini telah digunakan pada praktikum sebelumnya. Detail fitur dapat Anda pelajari pada link yang tersedia.

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [1]:
url = "https://github.com/adikara-ub/praktikum-ai-lanjut/raw/main/dataset/car_sample.csv"

Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [2]:
import pandas as pd
import numpy as np
data = pd.read_csv(url)



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [3]:
data.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


#### 2) Membagi data menjadi data latih dan data uji

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**. Agar pengacakan data dilakukan secara konstan, parameter **random_state** diisi dengan nilai integer tertentu, pada praktikum ini diset 101. Kemudian, nilai indeks pada data latih dan data uji diatur ulang agar berurutan nilainya


In [4]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data, test_size=0.2, random_state=101)
data_latih.reset_index(drop=True)
data_uji.reset_index(drop=True)

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,high,4,more,med,med,unacc
1,low,med,2,more,small,high,unacc
2,vhigh,low,5more,2,big,med,unacc
3,low,vhigh,3,more,small,med,unacc
4,med,low,3,more,small,low,unacc
...,...,...,...,...,...,...,...
341,high,med,2,more,med,med,unacc
342,low,med,4,more,big,med,good
343,vhigh,vhigh,3,2,big,med,unacc
344,vhigh,low,2,4,med,low,unacc


Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 208 data, dan **data_uji** terdiri dari 52 data

In [5]:
print(data_uji.shape[0])
print(data_latih.shape[0])

346
1382


#### 3) Menghitung Gini

Nilai Gini merupakan salah satu kriteria penentu variabel apa yang akan digunakan untuk membentuk cabang pada decision tree. Variabel dengan nilai Gini terbesar akan digunakan sebagai pembentukan cabang

Buatlah fungsi bernama **hitung_gini** yang berfungsi menghitung nilai Gini dari suatu nilai pada sebuah variabel

In [6]:
def hitung_gini(kolom_kelas):
    elemen, banyak = np.unique(kolom_kelas, return_counts=True)
    nilai_gini = 1 - np.sum([(banyak[i] / np.sum(banyak)) ** 2 for i in range(len(elemen))])
    return nilai_gini

Buatlah fungsi bernama **gini_split** yang digunakan untuk menghitung nilai Gini keseluruhan dari sebuah variabel.

In [7]:
def gini_split(data, nama_fitur_split, nama_fitur_kelas):
    nilai, banyak = np.unique(data[nama_fitur_split], return_counts=True)
    gini_split = np.sum([(banyak[i] / np.sum(banyak)) * hitung_gini(data.where(
        data[nama_fitur_split]==nilai[i]).dropna()[nama_fitur_kelas]) for i in 
        range(len(nilai))])
    return gini_split

Ujilah fungsi **gini_split** menggunakan data_latih pada variabel **buying** dan variabel kelas bernama **class**.

In [8]:
print(gini_split(data_latih, "buying", "class"))

0.4498424838345615


#### 4) Pembentukan pohon

Pembentukan pohon dilakukan secara rekursif. Seperti metode rekursif pada umumnya, perlu ditentukan kondisi berhenti terlebih dahulu. Kondisi berhenti pada pembentukan pohon adalah:


1.   Jika hanya ada satu kelas pada data, kembalikan kelas tersebut
2.   Jika fitur data  = 0 (tidak ada fitur yang tersisa), kembalikan kelas dari parent
3. Jika data kosong (tidak ada data), kembalikan kelas dengan frekuensi terbanyak

Selain kondisi berhenti tersebut, dilakukan pembentukan pohon secara rekursif menggunakan fungsi **buat_tree**.



In [9]:
def buat_tree(data, data_awal, daftar_fitur, nama_fitur_kelas,kelas_parent_node=None):
    #jika hanya ada satu kelas pada data
    if len(np.unique(data[nama_fitur_kelas])) <= 1:
        return np.unique(data[nama_fitur_kelas])[0]
    
    #jika data kosong
    elif len(data) == 0:
        return np.unique(data_awal[nama_fitur_kelas])[np.argmax(np.unique(data_awal[nama_fitur_kelas],return_counts=True)[1])]
    
    #jika tidak ada fitur yang terisa
    elif len(daftar_fitur) == 0:
        return kelas_parent_node
    
    else:
        kelas_parent_node = np.unique(data[nama_fitur_kelas])[np.argmax(np.unique(data[nama_fitur_kelas],return_counts=True)[1])]
        nilai_split = [gini_split(data, fitur, nama_fitur_kelas) for fitur in daftar_fitur]
        index_fitur_terbaik = np.argmin(nilai_split)
        fitur_terbaik = daftar_fitur[index_fitur_terbaik]
        tree = {fitur_terbaik:{}}
        daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
        for nilai in np.unique(data[fitur_terbaik]):
            sub_data = data.where(data[fitur_terbaik] == nilai).dropna()
            subtree = buat_tree(sub_data,data_awal, daftar_fitur,nama_fitur_kelas,kelas_parent_node)
            tree[fitur_terbaik][nilai]=subtree
    return(tree)

Buatlah tree menggunakan data latih yang tersedia

In [10]:
tree = buat_tree(data_latih, data_latih, data_latih.columns[:-1], "class")

Tampilkan tree yang terbentuk. Gunakan library **pprint** untuk menampilkan dictionary secara teratur.

In [11]:
from pprint import pprint
pprint(tree)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood'}},
                                                                                          'small': 'acc'}},
                                    

#### 5) Proses Prediksi

Proses prediksi kelas pada data uji dilakukan dengan melakukan *tree traversal* sampai menemui leaf.

In [12]:
def prediksi(data_uji,tree):
    for key in list(data_uji.keys()):
        if key in list(tree.keys()):
            try:
                hasil = tree[key][data_uji[key]]
            except:
                return 1
            
            hasil = tree[key][data_uji[key]]
            if isinstance(hasil,dict):
                return prediksi(data_uji,hasil)
            else:
                return hasil

#### 6) Proses Pengujian
Lakukan pengujian menggunakan data uji. Kelas pada data uji perlu dihapus dan data uji perlu diubah menjadi dictionary

In [13]:
data_uji_dict = data_uji.iloc[:,:-1].to_dict(orient='records')

Lakukan pengujian terhadap keseluruhan data uji menggunakan looping.

In [14]:
hasil_prediksi_total = []
for i in range(len(data_uji_dict)):
    hasil_prediksi = prediksi(data_uji_dict[i],tree)
    hasil_prediksi_total.append(hasil_prediksi)

Bandingkan hasil prediksi dengan label sebenarnya. Hitunglah banyaknya data uji yang memiliki kelas prediksi sama dengan kelas sebenarnya

In [15]:
print("Total prediksi benar: ", sum(hasil_prediksi_total == data_uji['class']))
print("Total prediksi salah: ", sum(hasil_prediksi_total != data_uji['class']))
print("Total data: ", len(data_uji))

Total prediksi benar:  298
Total prediksi salah:  48
Total data:  346


## XGBoost

#### 1) Instalasi module/package/library XGBoost

In [16]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


#### 2) Impor Data

In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(url)

#### 3) Pembuatan Classifier dan Model

In [18]:
label_encoder = LabelEncoder()
df_new = df.apply(label_encoder.fit_transform)

X_train, X_test, y_train, y_test = train_test_split(df_new.loc[:, 'buying': 'safety' ], df_new['class'], test_size=0.2)

# create model instance
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary: logistic' )

#### 4) Proses Pengujian/Prediksi

In [19]:
# fit model
bst.fit(X_train, y_train)

# make predictions
y_pred = bst.predict(X_test)

In [20]:
print(y_pred)

[2 2 0 2 0 0 2 0 2 0 2 2 2 0 0 2 0 2 2 2 2 2 0 2 0 2 2 2 2 2 2 0 2 0 2 2 2
 2 2 2 0 2 2 2 2 0 0 2 2 2 2 2 0 2 2 0 0 0 2 2 0 2 0 2 2 0 2 0 0 0 2 2 2 2
 0 2 0 2 0 0 2 2 0 2 2 2 2 0 0 0 0 2 0 0 2 0 0 2 0 2 0 2 0 0 0 2 0 2 2 0 2
 0 0 0 2 2 2 2 0 2 2 0 0 2 0 0 0 2 2 2 2 2 2 2 0 2 0 2 2 0 2 2 2 2 2 2 2 2
 0 0 2 0 0 2 0 2 2 0 2 0 0 2 2 2 0 2 2 2 2 2 2 2 0 2 0 0 0 2 0 0 2 2 0 2 2
 0 2 0 2 2 2 0 2 0 2 2 2 2 0 2 0 2 2 2 2 0 0 0 0 2 2 0 2 0 2 0 0 2 2 0 2 2
 0 0 2 0 2 0 0 2 0 0 2 0 2 0 0 0 0 0 2 0 2 2 2 0 2 2 0 2 0 0 2 0 0 2 0 0 0
 0 0 2 2 2 0 2 2 2 0 0 0 0 0 2 2 0 0 2 2 0 0 2 2 0 2 2 0 2 2 0 2 0 0 0 2 2
 0 0 2 2 0 2 0 0 2 2 0 0 0 2 0 2 0 2 2 0 0 0 0 2 0 2 0 0 2 2 2 0 0 2 2 0 2
 0 2 2 2 0 0 2 2 0 0 2 2 2]


In [21]:
y_label = label_encoder.inverse_transform(y_test)
print(y_label)

['unacc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc'
 'acc' 'acc' 'acc' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc'
 'unacc' 'unacc' 'good' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc'
 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'good' 'unacc'
 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'good' 'unacc' 'unacc'
 'unacc' 'unacc' 'good' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'unacc'
 'vgood' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'acc'
 'unacc' 'vgood' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'vgood' 'vgood' 'acc' 'unacc' 'unacc' 'unacc'
 'un

## TUGAS
Pada tugas kali ini Anda diminta memodifikasi metode pembentukan tree yang telah Anda agar metode tersebut menggunakan information gain sebagai dasar percabangan. Lengkapilah kerangka source code di bawah ini

Lengkapi fungsi hitung_entropy

In [22]:
def hitung_entropy(kolom_kelas):
  if (len(kolom_kelas) == 0):
    return 0
  
  _, banyak = np.unique(kolom_kelas, return_counts=True)
  probabilitas = banyak / len(kolom_kelas)
  probabilitas = probabilitas[probabilitas > 0]

  entropy = -np.sum(probabilitas * np.log2(probabilitas))
  return entropy

Lengkapi fungsi information_gain

In [23]:
def information_gain(data, nama_fitur_split, nama_fitur_kelas):
  entropy_awal = hitung_entropy(data[nama_fitur_kelas])
  nilai_unik = data[nama_fitur_split].unique()
  total = len(data)

  weighted_entropy = 0
  for nilai in nilai_unik:
        subset = data[data[nama_fitur_split] == nilai][nama_fitur_kelas]
        bobot = len(subset) / len(data)
        weighted_entropy += bobot * hitung_entropy(subset)
  
  information_gain = entropy_awal - weighted_entropy
  return information_gain

Lengkapi fungsi **buat_tree_ig**. Isinya sama persis dengan fungsi **buat_tree**, hanya saja penghitungan **gini_split** diganti dengan **information_gain**. Selain itu, percabangan dilakukan dengan menggunakan nilai **information_gain** **terbesar**

In [24]:
def buat_tree_ig(data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node=None):
  #jika hanya ada satu kelas pada data
  if len(np.unique(data[nama_fitur_kelas])) <= 1:
      return np.unique(data[nama_fitur_kelas])[0]
  
  #jika data kosong
  elif len(data) == 0:
      return np.unique(data_awal[nama_fitur_kelas])[np.argmax(np.unique(data_awal[nama_fitur_kelas],return_counts=True)[1])]
  
  #jika tidak ada fitur yang terisa
  elif len(daftar_fitur) == 0:
      return kelas_parent_node
  
  else:
      kelas_parent_node = np.unique(data[nama_fitur_kelas])[np.argmax(np.unique(data[nama_fitur_kelas],return_counts=True)[1])]
      nilai_split = [information_gain(data, fitur, nama_fitur_kelas) for fitur in daftar_fitur]
      index_fitur_terbaik = np.argmax(nilai_split)
      fitur_terbaik = daftar_fitur[index_fitur_terbaik]
      tree = {fitur_terbaik:{}}
      daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
      for nilai in np.unique(data[fitur_terbaik]):
          sub_data = data.where(data[fitur_terbaik] == nilai).dropna()
          subtree = buat_tree_ig(sub_data,data_awal, daftar_fitur,nama_fitur_kelas,kelas_parent_node)
          tree[fitur_terbaik][nilai]=subtree
  return(tree)

Lakukan pembentukan tree menggunakan fungsi **buat_tree_ig**

In [25]:
tree_ig = buat_tree_ig(data_latih,data_latih,data_latih.columns[:-1],'class')

Tampilkan tree yang terbentuk

In [26]:
pprint(tree_ig)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood'}},
                                                                                          'small': 'acc'}},
                                    

Lakukan pengujian menggunakan tree yang terbentuk

In [27]:
hasil_prediksi_total_ig = []
for i in range(len(data_uji_dict)):
  hasil_prediksi = prediksi(data_uji_dict[i],tree_ig)
  hasil_prediksi_total_ig.append(hasil_prediksi)
print("Total prediksi benar: ",sum(hasil_prediksi_total_ig==data_uji['class']))

Total prediksi benar:  298


### PERTANYAAN

Jawablah pertanyaan di bawah ini



1.   Amati tree yang dihasilkan dengan kriteria percabangan GINI dan Information Gain. Apa perbedaan tree yang dihasilkan dari kedua metode tersebut?
2.   Apakah penggunaan Information Gain dapat meningkatkan akurasi prediksi?



Tulis jawaban Anda di cell ini

1.   Jika diamati, terdapat satu perbedaan struktural pada cabang safety=low, persons=more, dan buying=high. Pada Gini, split berikutnya adalah maint terlebih dahulu, namun pada Information Gain, split berikutnya adalah lug_boot. Artinya pada subtree tersebut, Gini menganggap maint lebih penting untuk di-split lebih awal, sedangkan Information Gain menganggap lug_boot lebih penting.

2.   Penggunaan Information Gain tidak menjamin dapat meningkatkan akurasi prediksi dibandingkan Gini. Keduanya memiliki tujuan yang sama dalam meminimalkan impurity, sehingga sering kali menghasilkan akurasi prediksi yang mirip dan setara. 

